In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cm_to_inch = 1 / 2.54  # centimeters in inches

In [ ]:
# Plot data for the first term
def calc_global_term(corr, a=25):
    term_global = a * np.tan(-0.5 * np.pi * corr)
    return term_global


def calc_cons_term(frac, b=0.3, c=10):
    term_cons = -((frac - b) ** c)
    return term_cons


xx_corr = np.linspace(-0.9, 0.9, 100)
yy_corr = calc_global_term(xx_corr)
yy_corr_a2 = calc_global_term(xx_corr, a=10)

xx_cons = np.linspace(0.0, 2, 100)
yy_cons = calc_cons_term(xx_cons)
yy_cons_b2 = calc_cons_term(xx_cons, b=0.0)

# Make the 'image' for total
rew_img = np.zeros((100, 100))
for i1, xcr in enumerate(xx_corr):
    for i2, xcn in enumerate(xx_cons):
        rew_img[i1][i2] = (
            calc_cons_term(xcn, b=0.23) / 100 + calc_global_term(xcr, a=30) / 100
        )

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# The idea here is to generate images for the reward function of the RL
fig = plt.figure(figsize=(9 * cm_to_inch, 15 * cm_to_inch))
gs = fig.add_gridspec(3, 2)
ax_corr = fig.add_subplot(gs[0])
ax_cons = fig.add_subplot(gs[1])
ax_tot = fig.add_subplot(gs[2:], aspect="equal")
cbar_ax = inset_axes(
    ax_tot,
    width="30%",
    height="3%",
    bbox_to_anchor=(0.05, 0.4, 0.6, 0.55),
    bbox_transform=ax_tot.transAxes,
    loc="upper left",
)
# Plot corr term
ax_corr.axvline(x=0.0, c="k", alpha=0.4, linestyle="--")
ax_corr.plot(xx_corr, yy_corr / 100, label="$C_1=25$")
ax_corr.plot(xx_corr, yy_corr_a2 / 100, label="$C_1=10$")
ax_corr.set(ylim=(-1, 1), xlim=(-0.9, 0.9))
ax_corr.set_xticks([-0.9, 0.0, 0.9])
ax_corr.grid(alpha=0.4)
ax_corr.set(xlabel="$\\text{PC}(P^{grid}_{n}, P^{grid}_{\sim{n}})$", title="$r^{peak}$")
ax_corr.legend(fontsize=8, handlelength=1)
# Plot cons term
ax_cons.axvline(x=1.0, c="k", alpha=0.4, linestyle="--")
ax_cons.plot(xx_cons, yy_cons / 100, label="$C_2=0.3$")
ax_cons.plot(xx_cons, yy_cons_b2 / 100, label="$C_2=0.0$")
ax_cons.grid(alpha=0.4)
ax_cons.set(
    xlabel="$E^{rl}_n/E^{b}_n$", title="$r^{cons}$", xlim=(0.5, 2), ylim=(-1, 0.1)
)
ax_cons.legend(fontsize=8, handlelength=1)
for ax in (ax_corr, ax_cons):
    ax.set_yticks([-1.0, 0.0, 1.0])
    ax.set_yticklabels(["Min", "0", "Max"], fontsize=8)
ax_cons.set_yticklabels([])
# Plot total image
im = ax_tot.imshow(rew_img, cmap="jet")
ax_tot.set_title("$r_t = r^{peak} + r^{cons}$", fontsize=10)
ax_tot.set_xlabel("$\\text{PC}(P^{grid}_{n}, P^{grid}_{\sim{n}})$", fontsize=9)
ax_tot.set_ylabel("$E^{rl}_n/E^{b}_n$", fontsize=9)
ax_tot.set(xlim=(0, 99), xticks=[0, 49, 99], xticklabels=["-0.9", "0.0", "0.9"])
ax_tot.set(ylim=(0, 99), yticks=[0, 49, 99], yticklabels=["0.0", "1.0", "2.0"])
ax_tot.axhline(y=49, linestyle="--", c="k", alpha=0.5)
ax_tot.axvline(x=49, linestyle="--", c="k", alpha=0.5)
# arrow = FancyArrowPatch((74, 74), (24, 24), mutation_scale=15, color="g")
# ax_tot.add_patch(arrow)
ax_tot.text(
    24,
    24,
    "Less Energy\nLess Correlation\nHigher Reward",
    ha="center",
    va="center",
    fontsize=9,
)
ax_tot.text(
    74,
    74,
    "More Energy\nMore Correlation\nLower Reward",
    ha="center",
    va="center",
    fontsize=9,
)
# Add a small colorbar inside ax_tot
cbar = fig.colorbar(
    im, ax=ax_tot, cax=cbar_ax, orientation="horizontal", fraction=0.05, pad=0.1
)
cbar.set_ticks([rew_img.min(), rew_img.max()])
cbar.set_ticklabels(["Low", "High"])
cbar.ax.tick_params(labelsize=8)
fig.tight_layout()
# fig.savefig("export/rew_func.png", dpi=400, bbox_inches="tight")

In [ ]:
import matplotlib as mpl

new_rc_params = {"text.usetex": False, "svg.fonttype": "none"}
mpl.rcParams.update(new_rc_params)
fig.savefig("figures/rew_func.svg", bbox_inches="tight")